# Synthetic Data Generation Tutorial using phi4, llama3, and mixtral

This tutorial demonstrates how to use SDG repository to generate synthetic question-answer pairs from documents using large language models like phi4 and llama3. We will also generate data using mixtral model for comparison. We'll cover:

1. Setting up the environment
2. Connecting to LLM servers
3. Configuring the data generation pipeline
4. Generating data with different models
5. Comparing results

In [ ]:
# Enable auto-reloading of modules - useful during development
%load_ext autoreload
%autoreload 2

## Setup Instructions

Before running this notebook, you'll need to:

```bash 
pip install sdg-hub==0.1.0a4
```

In [ ]:
%%capture
%pip install transformers

In [ ]:
# Import required libraries
# datasets: For handling our data
# OpenAI: For interfacing with the LLM servers
# SDG components: For building our data generation pipeline
from datasets import load_dataset, Dataset
from openai import OpenAI
from transformers import AutoTokenizer

from sdg_hub.flow import Flow
from sdg_hub.sdg import SDG
from sdg_hub.registry import PromptRegistry

In [ ]:
import datetime

now = datetime.datetime.now()
timestamp = now.strftime('%Y%m%d-%H%M%S')

In [ ]:
force_ascii = True  # NOTE this is default
# force_ascii = False

### Configure Parallelism

In [ ]:
# For production
num_workers = 8   # Number of parallel workers
batch_size = 8    # Batch size for processing
save_freq = 1000  # How often to save checkpoints

# For test
# num_workers = 1   # Number of parallel workers
# batch_size = 1    # Batch size for processing
# save_freq = 1000  # How often to save checkpoints

### Setup environments for [RITS](https://rits.fmaas.res.ibm.com/)

In [ ]:
import os
import requests

RITS_API_KEY = os.getenv("RITS_API_KEY")
# print(f"RITS_API_KEY={RITS_API_KEY}", flush=True)

default_headers = {"RITS_API_KEY": RITS_API_KEY}

url = "https://rits.fmaas.res.ibm.com/ritsapi/inferenceinfo"
res = requests.get(url=url, headers=default_headers)
assert res.status_code == 200
model_list: list[dict[str, str]] = res.json()
model_dict = { m["model_name"]: m["endpoint"] for m in model_list }
# NOTE avoid clashes in model_name
model_dict["meta-llama/llama-3-3-70b-instruct"] = "https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/llama-3-3-70b-instruct"
model_dict["microsoft/phi-4"] = "https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/microsoft-phi-4"
model_dict["mistralai/mixtral-8x7B-instruct-v0.1"] = "https://inference-3scale-apicast-production.apps.rits.fmaas.res.ibm.com/mixtral-8x7b-instruct-v01"

def get_base_url(model_name: str)-> str:
    endpoint = model_dict.get(model_name, "http://0.0.0.0:8000")  # fall back to vllm
    return f"{endpoint}/v1"

### Configure Seed Data

In [ ]:
# data_name = "20250411_en_2"
# data_name = "20250411_ja"
# data_name = "teigaku-genzei"
# data_name = "teigaku-genzei-ibm-v0"
# data_name = "teigaku-genzei-ibm-v2"
data_name = "teigaku-genzei-ibm-v3"

if "20250411_ja" in data_name or "teigaku-genzei" in data_name:
    data_lang = "_ja"
else:
    data_lang = ""

seed_data_name = f"seed_data_{data_name}"
seed_data_path = f"{seed_data_name}.jsonl"

In [ ]:
repeat_times = 1
# repeat_times = 10

data_name_repeat = f"{data_name}-r{repeat_times}" if repeat_times > 1 else data_name

### Load and Prepare Seed Data

We'll load our seed data (documents) that will be used to generate question-answer pairs.

In [ ]:
# Load the seed data from JSON file
ds = load_dataset('json', data_files=seed_data_path, split='train')

### (Optional) Repeat Seed Data

In [ ]:
if repeat_times > 1:
    ds = ds.repeat(repeat_times)

In [ ]:
print(f"Loaded {len(ds)} seed data", flush=True)

### Utilities for Generated Data

In [ ]:
def to_messages(generated_data: Dataset) -> Dataset:
    messages_list: list[dict[str, any]] = []
    for generated_datum in generated_data:
        user = generated_datum['question']
        assistant = generated_datum['response']
        messages = [
            {"role": "user", "content": user},
            {"role": "assistant", "content": assistant},
        ]
        messages_list.append({"messages": messages})
    messages_data = Dataset.from_list(messages_list)
    return messages_data

def get_dataset_type(generated_data_i: dict[str, any]) -> str:
    _dataset_type = generated_data_i.get('dataset_type', None)
    if _dataset_type is not None:
        _dataset_type = f" ({_dataset_type})"
    else:
        _dataset_type = ""
    return _dataset_type

def write_input(f, generated_data_i) -> None:
    icl_document = generated_data_i.get('icl_document', None)
    if icl_document is not None:
        f.write(f"### In-Context Learning Example\n\n")
        f.write(f"#### ICL Document\n")
        f.write(icl_document + "\n\n")
    icl_query_1 = generated_data_i.get('icl_query_1', None)
    if icl_query_1 is not None:
        f.write(f"#### ICL Query 1\n")
        f.write(icl_query_1 + "\n\n")
    icl_response_1 = generated_data_i.get('icl_response_1', None)
    if icl_response_1 is not None:
        f.write(f"#### ICL Response 1\n")
        f.write(icl_response_1 + "\n\n")
    icl_query_2 = generated_data_i.get('icl_query_2', None)
    if icl_query_2 is not None:
        f.write(f"#### ICL Query 2\n")
        f.write(icl_query_2 + "\n\n")
    icl_response_2 = generated_data_i.get('icl_response_2', None)
    if icl_response_2 is not None:
        f.write(f"#### ICL Response 2\n")
        f.write(icl_response_2 + "\n\n")
    icl_query_3 = generated_data_i.get('icl_query_3', None)
    if icl_query_3 is not None:
        f.write(f"#### ICL Query 3\n")
        f.write(icl_query_3 + "\n\n")
    icl_response_3 = generated_data_i.get('icl_response_3', None)
    if icl_response_3 is not None:
        f.write(f"#### ICL Response 3\n")
        f.write(icl_response_3 + "\n\n")
    document_outline = generated_data_i.get('document_outline', None)
    if document_outline is not None:
        f.write(f"### Document Outline\n")
        f.write(document_outline + "\n\n")
    raw_document = generated_data_i.get('raw_document', None)
    if raw_document is not None:
        f.write(f"### Raw Document (not used for Q&A generation)\n")
        f.write(raw_document + "\n\n")

### Select Models

In [ ]:
generate_data_with_phi4 = True
generate_data_with_llama3 = False
generate_data_with_mixtral = False

## SDG with phi4 Model

### Setting up phi4 Model

In [ ]:
if generate_data_with_phi4:
    # Connect to phi4 model running on RITS
    phi4_teacher_model = "microsoft/phi-4"
    phi4_endpoint = get_base_url(phi4_teacher_model)

    phi4_client = OpenAI(
        api_key="EMPTY",
        base_url=phi4_endpoint,
        default_headers=default_headers,
    )

    # Verify connection to phi4 model
    print(f"Connected to phi4 model: {phi4_teacher_model}", flush=True)

### Configure phi4 Prompt Template

In [ ]:
if generate_data_with_phi4:
    # Register the phi4 chat template
    # This ensures proper formatting of prompts for the model

    phi4_teacher_model_hf = "microsoft/phi-4"

    # Load the tokenizer to get the chat template
    phi4_tokenizer = AutoTokenizer.from_pretrained(phi4_teacher_model_hf)

    # Register the chat template in our prompt registry
    @PromptRegistry.register(phi4_teacher_model)
    def phi4_chat_template():
        return phi4_tokenizer.chat_template

### Configure phi4 Pipeline

In [ ]:
if generate_data_with_phi4:
    # Create flow configuration for phi4
    flow_phi4 = Flow(phi4_client).get_flow_from_file(f"synth_knowledge1.5{data_lang}_phi4_rits.yaml")

    # Initialize SDG pipeline for phi4
    sdg_phi4 = SDG(
        [flow_phi4],
        num_workers=num_workers,
        batch_size=batch_size,
        save_freq=save_freq,
    )

### Generate Data with phi4

In [ ]:
if generate_data_with_phi4:
    # Generate data using phi4 model
    generated_data_phi4 = sdg_phi4.generate(ds, checkpoint_dir=f"Tmp_{data_name_repeat}_phi4")

    generated_path_phi4 = f"generated_data_{data_name_repeat}_{timestamp}_phi4.jsonl"
    generated_data_phi4.to_json(generated_path_phi4, orient="records", lines=True, force_ascii=force_ascii)
    print(f"Data saved to {generated_path_phi4}", flush=True)

    # Save generated data in messages format for training
    messages_data_phi4 = to_messages(generated_data_phi4)

    messages_data_path_phi4 = f"messages_data_{data_name_repeat}_{timestamp}_phi4.jsonl"
    messages_data_phi4.to_json(messages_data_path_phi4, orient="records", lines=True, force_ascii=force_ascii)
    print(f"Messages data saved to {messages_data_path_phi4}", flush=True)

### Output Generated Data with phi4

In [ ]:
if generate_data_with_phi4:
    # Save comparison results to markdown file
    output_file = f"model_output_{data_name_repeat}_{timestamp}_phi4.md"

    if 'generated_data_phi4' not in locals():
        generated_data_phi4 = []

    with open(output_file, "w") as f:
        num_generated_data_phi4 = len(generated_data_phi4)

        # Number of examples to compare
        k = num_generated_data_phi4

        # Compare generated Q&A pairs
        for i in range(k):
            f.write("# Example #{}\n\n".format(i+1))

            if i < num_generated_data_phi4:
                # phi4 results
                write_input(f, generated_data_phi4[i])
                f.write(f"### Document{get_dataset_type(generated_data_phi4[i])} from phi4\n")
                f.write(generated_data_phi4[i]['document'] + "\n\n")
                f.write("### Result from phi4\n")
                f.write(generated_data_phi4[i]['question'] + "\n")
                f.write("*******************************\n")
                f.write(generated_data_phi4[i]['response'] + "\n")

            f.write("\n")

    print(f"Wrote {k} examples to {output_file}", flush=True)

## (Optional) SDG with llama3 Model

### Setting up llama3 Model

In [ ]:
if generate_data_with_llama3:
    # Configure OpenAI client to connect to RITS server
    llama3_teacher_model = "meta-llama/llama-3-3-70b-instruct"
    llama3_endpoint = get_base_url(llama3_teacher_model)

    llama3_client = OpenAI(
        api_key="EMPTY",
        base_url=llama3_endpoint,
        default_headers=default_headers,
    )

    print(f"Connected to llama3 model: {llama3_teacher_model}", flush=True)

### Configure llama3 Prompt Template

We need to register the correct chat template for our model to ensure proper prompt formatting.

In [ ]:
if generate_data_with_llama3:
    # Register the llama3 chat template
    # This ensures proper formatting of prompts for the model

    # llama3_teacher_model_hf = "meta-llama/Llama-3.3-70B-Instruct"
    llama3_teacher_model_hf = "unsloth/Llama-3.3-70B-Instruct"

    # Load the tokenizer to get the chat template
    llama3_tokenizer = AutoTokenizer.from_pretrained(llama3_teacher_model_hf)

    # Register the chat template in our prompt registry
    @PromptRegistry.register(llama3_teacher_model)
    def llama3_chat_template():
        return llama3_tokenizer.chat_template

### Configure the Data Generation Pipeline

Now we'll set up our Synthetic Data Generation (SDG) pipeline with the following components:
1. SDG Flow configuration from YAML
2. SDG Pipeline setup
3. SDG configuration with batch processing, number of workers, and save frequency parameters

In [ ]:
if generate_data_with_llama3:
    # Load the flow configuration from YAML file
    flow_llama3 = Flow(llama3_client).get_flow_from_file(f"synth_knowledge1.5{data_lang}_llama3_rits.yaml")

    # Initialize the SDG pipeline with processing parameters
    sdg_llama3 = SDG(
        [flow_llama3],
        num_workers=num_workers,
        batch_size=batch_size,
        save_freq=save_freq,
    )

### Generate Data with llama3

Now we'll use our configured pipeline to generate synthetic question-answer pairs.

In [ ]:
if generate_data_with_llama3:
    # Generate synthetic data and save checkpoints
    generated_data_llama3 = sdg_llama3.generate(ds, checkpoint_dir=f"Tmp_{data_name_repeat}_llama3")

    generated_path_llama3 = f"generated_data_{data_name_repeat}_{timestamp}_llama3.jsonl"
    generated_data_llama3.to_json(generated_path_llama3, orient="records", lines=True, force_ascii=force_ascii)
    print(f"Data saved to {generated_path_llama3}", flush=True)

    # Save generated data in messages format for training
    messages_data_llama3 = to_messages(generated_data_llama3)

    messages_data_path_llama3 = f"messages_data_{data_name_repeat}_{timestamp}_llama3.jsonl"
    messages_data_llama3.to_json(messages_data_path_llama3, orient="records", lines=True, force_ascii=force_ascii)
    print(f"Messages data saved to {messages_data_path_llama3}", flush=True)

### Output Generated Data with llama3

In [ ]:
if generate_data_with_llama3:
    # Save comparison results to markdown file
    output_file = f"model_output_{data_name_repeat}_{timestamp}_llama3.md"

    if 'generated_data_llama3' not in locals():
        generated_data_llama3 = []

    with open(output_file, "w") as f:
        num_generated_data_llama3 = len(generated_data_llama3)

        # Number of examples to compare
        k = num_generated_data_llama3

        # Compare generated Q&A pairs
        for i in range(k):
            f.write("# Example #{}\n\n".format(i+1))

            if i < num_generated_data_llama3:
                # LLaMA 3.3 results
                write_input(f, generated_data_llama3[i])
                f.write(f"### Document{get_dataset_type(generated_data_llama3[i])} from llama3\n")
                f.write(generated_data_llama3[i]['document'] + "\n\n")
                f.write("### Result from llama3\n")
                f.write(generated_data_llama3[i]['question'] + "\n")
                f.write("*******************************\n")
                f.write(generated_data_llama3[i]['response'] + "\n")

            f.write("\n")

    print(f"Wrote {k} examples to {output_file}", flush=True)

## (Optional) SDG with mixtral Model

### Setting up mixstal Model

For comparison, we'll also generate data using the mixtral model.

In [ ]:
if generate_data_with_mixtral:
    # Connect to mixtral model running on RITS
    mixtral_teacher_model = "mistralai/mixtral-8x7B-instruct-v0.1"
    mixtral_endpoint = get_base_url(mixtral_teacher_model)

    mixtral_client = OpenAI(
        api_key="EMPTY",
        base_url=mixtral_endpoint,
        default_headers=default_headers,
    )

    # Verify connection to mixtral model
    print(f"Connected to mixtral model: {mixtral_teacher_model}", flush=True)

### Configure mixtral Prompt Template

We need to register the correct chat template for our model to ensure proper prompt formatting.

In [ ]:
if generate_data_with_mixtral:
    # Register the mixtral chat template
    # This ensures proper formatting of prompts for the model

    mixtral_teacher_model_hf = "mistralai/Mixtral-8x7B-Instruct-v0.1"

    # Load the tokenizer to get the chat template
    mixtral_tokenizer = AutoTokenizer.from_pretrained(mixtral_teacher_model_hf)

    # Register the chat template in our prompt registry
    @PromptRegistry.register(mixtral_teacher_model)
    def mixtral_chat_template():
        return mixtral_tokenizer.chat_template

### Configure mixtral Pipeline

Set up a similar pipeline for mixtral model generation.

In [ ]:
if generate_data_with_mixtral:
    # Create flow configuration for mixtral
    flow_mixtral = Flow(mixtral_client).get_flow_from_file(f"synth_knowledge1.5{data_lang}_mixtral_rits.yaml")

    # Initialize SDG pipeline for mixtral
    sdg_mixtral = SDG(
        [flow_mixtral],
        num_workers=num_workers,
        batch_size=batch_size,
        save_freq=save_freq,
    )

### Generate Data with mixtral

Generate synthetic data using the mixtral model for comparison.

In [ ]:
if generate_data_with_mixtral:
    # Generate data using mixtral model
    generated_data_mixtral = sdg_mixtral.generate(ds, checkpoint_dir=f"Tmp_{data_name_repeat}_mixtral")

    generated_path_mixtral = f"generated_data_{data_name_repeat}_{timestamp}_mixtral.jsonl"
    generated_data_mixtral.to_json(generated_path_mixtral, orient="records", lines=True, force_ascii=force_ascii)
    print(f"Data saved to {generated_path_mixtral}", flush=True)

    # Save generated data in messages format for training
    messages_data_mixtral = to_messages(generated_data_mixtral)

    messages_data_path_mixtral = f"messages_data_{data_name_repeat}_{timestamp}_mixtral.jsonl"
    messages_data_mixtral.to_json(messages_data_path_mixtral, orient="records", lines=True, force_ascii=force_ascii)
    print(f"Messages data saved to {messages_data_path_mixtral}", flush=True)

### Output Generated Data with mixtral

In [ ]:
if generate_data_with_mixtral:
    # Save comparison results to markdown file
    output_file = f"model_output_{data_name_repeat}_{timestamp}_mixtral.md"

    if 'generated_data_mixtral' not in locals():
        generated_data_mixtral = []

    with open(output_file, "w") as f:
        num_generated_data_mixtral = len(generated_data_mixtral)

        # Number of examples to compare
        k = num_generated_data_mixtral

        # Compare generated Q&A pairs
        for i in range(k):
            f.write("# Example #{}\n\n".format(i+1))

            if i < num_generated_data_mixtral:
                # mixtral results
                write_input(f, generated_data_mixtral[i])
                f.write(f"### Document{get_dataset_type(generated_data_mixtral[i])} from mixtral\n")
                f.write(generated_data_mixtral[i]['document'] + "\n\n")
                f.write("### Result from mixtral\n")
                f.write(generated_data_mixtral[i]['question'] + "\n")
                f.write("*******************************\n")
                f.write(generated_data_mixtral[i]['response'] + "\n")

            f.write("\n")

    print(f"Wrote {k} examples to {output_file}", flush=True)

## Compare Generated Data

Let's compare the outputs from both models by saving them to a markdown file for easy review.

In [ ]:
# Save comparison results to markdown file
output_file = f"model_comparison_{data_name_repeat}_{timestamp}.md"

if 'generated_data_phi4' not in locals():
    generated_data_phi4 = []

if 'generated_data_llama3' not in locals():
    generated_data_llama3 = []

if 'generated_data_mixtral' not in locals():
    generated_data_mixtral = []

with open(output_file, "w") as f:
    num_generated_data_phi4 = len(generated_data_phi4)
    num_generated_data_llama3 = len(generated_data_llama3)
    num_generated_data_mixtral = len(generated_data_mixtral)

    # Number of examples to compare
    k = max(num_generated_data_phi4, num_generated_data_llama3, num_generated_data_mixtral)

    # Compare generated Q&A pairs
    for i in range(k):
        f.write("# Example #{}\n\n".format(i+1))

        if i < num_generated_data_phi4:
            # phi4 results
            write_input(f, generated_data_phi4[i])
            f.write(f"### Document{get_dataset_type(generated_data_phi4[i])} from phi4\n")
            f.write(generated_data_phi4[i]['document'] + "\n\n")
            f.write("### Result from phi4\n")
            f.write(generated_data_phi4[i]['question'] + "\n")
            f.write("*******************************\n")
            f.write(generated_data_phi4[i]['response'] + "\n")

        if i < num_generated_data_llama3:
            # llama3 results
            write_input(f, generated_data_llama3[i])
            f.write(f"### Document{get_dataset_type(generated_data_llama3[i])} from llama3\n")
            f.write(generated_data_llama3[i]['document'] + "\n\n")
            f.write("### Result from llama3\n")
            f.write(generated_data_llama3[i]['question'] + "\n")
            f.write("*******************************\n")
            f.write(generated_data_llama3[i]['response'] + "\n")

        if i < num_generated_data_mixtral:
            # mixtral results
            write_input(f, generated_data_mixtral[i])
            f.write(f"### Document{get_dataset_type(generated_data_mixtral[i])} from mixtral\n")
            f.write(generated_data_mixtral[i]['document'] + "\n\n")
            f.write("### Result from mixtral\n")
            f.write(generated_data_mixtral[i]['question'] + "\n")
            f.write("*******************************\n")
            f.write(generated_data_mixtral[i]['response'] + "\n")

        f.write("\n")

print(f"Wrote {k} examples to {output_file}", flush=True)

## Production Usage

For large-scale data generation, use the command-line script instead of this notebook:

```bash
python scripts/generate.py --ds_path seed_data.jsonl \
    --bs 2 --num_workers 10 \
    --save_path <your_save_path> \
    --flow ../src/sdg_hub/flows/generation/knowledge/synth_knowledge1.5.yaml \
    --checkpoint_dir <your_checkpoint_dir> \
    --endpoint <your_endpoint>
```

Note: For LLaMA 3.3, use `synth_knowledge1.5_llama3.3.yaml` as the flow configuration file.